# Stage 3-4: ERAP2 Structure + Protein Binder Design

**Ancient Drug Discovery Pipeline**

This notebook:
1. Downloads and verifies the ERAP2 AlphaFold structure (pLDDT 93.31)
2. Checks PDB for experimental structures (cross-validation)
3. Visualizes the active site and variant positions
4. Runs Proteina-Complexa (NVIDIA) to generate binder candidates

**Target:** ERAP2 (Q6P179, 960 aa) — active site residues H370, E371, H374, E393

In [ ]:
# Cell 1: Install dependencies
!pip install -q biopython py3Dmol requests
import requests, json
print("Ready.")

In [ ]:
# Cell 2: Download ERAP2 structure from AlphaFold DB
import os

PDB_URL = "https://alphafold.ebi.ac.uk/files/AF-Q6P179-F1-model_v6.pdb"
PDB_FILE = "erap2_wt_alphafold.pdb"

if not os.path.exists(PDB_FILE):
    print("Downloading ERAP2 structure from AlphaFold DB...")
    resp = requests.get(PDB_URL)
    resp.raise_for_status()
    with open(PDB_FILE, "w") as f:
        f.write(resp.text)
    print(f"Downloaded: {len(resp.text)} characters")
else:
    print(f"Already have {PDB_FILE}")

# Verify
with open(PDB_FILE) as f:
    lines = f.readlines()
atom_lines = [l for l in lines if l.startswith("ATOM")]
residues = set()
for l in atom_lines:
    resnum = int(l[22:26].strip())
    residues.add(resnum)
print(f"PDB loaded: {len(atom_lines)} atoms, {len(residues)} residues")
print(f"Residue range: {min(residues)}-{max(residues)}")

# Check active site residues are present
active_site = {370: 'H', 371: 'E', 374: 'H', 393: 'E'}
for pos, aa in active_site.items():
    present = pos in residues
    print(f"  Active site {aa}{pos}: {'PRESENT' if present else 'MISSING'}")

In [ ]:
# Cell 3: Search PDB for experimental ERAP2 structures
PDB_SEARCH_URL = "https://search.rcsb.org/rcsbsearch/v2/query"
pdb_query = {
    "query": {
        "type": "group",
        "logical_operator": "and",
        "nodes": [
            {
                "type": "terminal",
                "service": "text",
                "parameters": {
                    "attribute": "rcsb_polymer_entity.pdbx_description",
                    "operator": "contains_words",
                    "value": "endoplasmic reticulum aminopeptidase 2"
                }
            }
        ]
    },
    "return_type": "entry"
}

resp = requests.post(PDB_SEARCH_URL, json=pdb_query)
if resp.ok:
    pdb_results = resp.json()
    hits = pdb_results.get("result_set", [])
    print(f"Found {len(hits)} experimental PDB structures for ERAP2")
    for hit in hits[:10]:
        pdb_id = hit.get("identifier", "")
        print(f"  {pdb_id} - https://www.rcsb.org/structure/{pdb_id}")
    if hits:
        print(f"\nThese can be used to cross-validate the AlphaFold prediction (RMSD < 2A target)")
else:
    print(f"PDB search returned {resp.status_code} — may need to check manually")

In [ ]:
# Cell 4: 3D visualization of ERAP2 structure
import py3Dmol

with open(PDB_FILE) as f:
    pdb_data = f.read()

view = py3Dmol.view(width=800, height=600)
view.addModel(pdb_data, "pdb")

# Cartoon representation colored by pLDDT (B-factor)
view.setStyle({"cartoon": {"color": "spectrum"}})

# Highlight active site residues in red spheres
for pos in [370, 371, 374, 393]:
    view.addStyle({"resi": pos}, {"stick": {"color": "red", "radius": 0.3}})
    view.addLabel(f"{pos}", {"fontSize": 10, "backgroundColor": "red", 
                              "fontColor": "white", "showBackground": True},
                 {"resi": pos, "atom": "CA"})

# Highlight K392 variant position in yellow
view.addStyle({"resi": 392}, {"stick": {"color": "yellow", "radius": 0.3}})
view.addLabel("K392N\n(variant)", {"fontSize": 10, "backgroundColor": "yellow",
                                    "fontColor": "black", "showBackground": True},
             {"resi": 392, "atom": "CA"})

view.zoomTo()
print("ERAP2 structure: spectrum = N(blue) to C(red)")
print("Red sticks = active site (H370, E371, H374, E393)")
print("Yellow stick = K392N variant position")
view.show()

In [ ]:
# Cell 5: Zoom into active site
view2 = py3Dmol.view(width=800, height=600)
view2.addModel(pdb_data, "pdb")

# Show only residues near the active site (within ~15A)
view2.setStyle({}, {"cartoon": {"color": "white", "opacity": 0.3}})

# Active site residues as sticks
for pos in [370, 371, 374, 393]:
    view2.addStyle({"resi": pos}, {"stick": {"colorscheme": "default", "radius": 0.2}})

# K392 variant
view2.addStyle({"resi": 392}, {"stick": {"color": "yellow", "radius": 0.2}})

# Surrounding residues as lines
for pos in range(365, 400):
    view2.addStyle({"resi": pos}, {"line": {"colorscheme": "default"}})

# Labels
view2.addLabel("H370 (Zn)", {"fontSize": 12, "fontColor": "red"}, {"resi": 370, "atom": "CA"})
view2.addLabel("E371 (cat)", {"fontSize": 12, "fontColor": "red"}, {"resi": 371, "atom": "CA"})
view2.addLabel("H374 (Zn)", {"fontSize": 12, "fontColor": "red"}, {"resi": 374, "atom": "CA"})
view2.addLabel("E393 (Zn)", {"fontSize": 12, "fontColor": "red"}, {"resi": 393, "atom": "CA"})
view2.addLabel("K392N", {"fontSize": 12, "fontColor": "orange"}, {"resi": 392, "atom": "CA"})

# Zoom to active site
view2.zoomTo({"resi": [370, 371, 374, 392, 393]})
print("Active site closeup — this is where binders need to fit")
view2.show()

---
## Stage 4: Protein Binder Design with Proteina-Complexa

**Proteina-Complexa** (NVIDIA, March 2026):
- 68% success rate across 127 diverse targets
- Picomolar binders (93.6 pM on PDGFR)
- Atomistic design — generates sequence + structure simultaneously
- Open source: https://github.com/NVIDIA-Digital-Bio/proteina-complexa

We target the ERAP2 active site: zinc-binding residues H370, H374, E393 and catalytic E371.

In [ ]:
# Cell 6: Install Proteina-Complexa
import os
if not os.path.exists("proteina-complexa"):
    !git clone https://github.com/NVIDIA-Digital-Bio/proteina-complexa.git
    %cd proteina-complexa
    !pip install -q -e . 2>&1 | tail -5
    %cd ..
else:
    print("Proteina-Complexa already cloned")

# Check if it installed correctly
try:
    import proteina
    print(f"Proteina-Complexa loaded successfully")
except ImportError as e:
    print(f"Import error: {e}")
    print("Check the repo README for exact install instructions")
    print("The package may need: pip install -e proteina-complexa/")

In [ ]:
# Cell 7: Define ERAP2 target for binder design
import torch

TARGET_PDB = "erap2_wt_alphafold.pdb"

# ERAP2 active site hotspot residues
# These are the residues we want the binder to contact
HOTSPOT_RESIDUES = [370, 371, 374, 392, 393]

print(f"Target: ERAP2 (Q6P179)")
print(f"Structure: {TARGET_PDB}")
print(f"Hotspot residues: {HOTSPOT_RESIDUES}")
print(f"  H370 - Zinc binding")
print(f"  E371 - Catalytic")
print(f"  H374 - Zinc binding")
print(f"  K392 - Variant position (K392N, balancing selection)")
print(f"  E393 - Zinc binding (HEXXH+E motif)")
print(f"")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
# Cell 8: Generate binder candidates
# NOTE: This cell depends on Proteina-Complexa's exact API.
# If the import above failed, check the repo README for the correct usage.
# The general pattern is:
#   1. Load the target PDB
#   2. Define hotspot residues
#   3. Run inference to generate binder sequences + structures
#   4. Score and rank candidates

# If Proteina-Complexa doesn't install cleanly, use RFdiffusion as fallback:
# !git clone https://github.com/RosettaCommons/RFdiffusion.git

# Or use Dyno Psi-1:
# !pip install dynopsi  # https://github.com/dynotx/dynopsi

print("Check Proteina-Complexa README for inference command.")
print("Expected output: ~100 binder candidates with sequences + structures")
print("")
print("Typical command pattern:")
print("  python proteina-complexa/scripts/inference.py \\")
print(f"    --target_pdb {TARGET_PDB} \\")
print(f"    --hotspot_residues {' '.join(str(r) for r in HOTSPOT_RESIDUES)} \\")
print("    --num_designs 100 \\")
print("    --output_dir binder_candidates/")

In [ ]:
# Cell 9: Quick drug-likeness check on any generated peptide binders
# This runs after binder candidates are generated

from rdkit import Chem
from rdkit.Chem import Descriptors

# Known ERAP2 inhibitor for comparison
BESTATIN_SMILES = "CC(O)C(=O)NC(CC1=CC=CC=C1)C(O)=O"  # DrugBank DB00560

mol = Chem.MolFromSmiles(BESTATIN_SMILES)
if mol:
    print("Reference: Bestatin (known aminopeptidase inhibitor)")
    print(f"  MW: {Descriptors.MolWt(mol):.1f}")
    print(f"  LogP: {Descriptors.MolLogP(mol):.2f}")
    print(f"  HBD: {Descriptors.NumHDonors(mol)}")
    print(f"  HBA: {Descriptors.NumHAcceptors(mol)}")
    print(f"  Lipinski: {'PASS' if Descriptors.MolWt(mol) <= 500 and Descriptors.MolLogP(mol) <= 5 else 'FAIL'}")
    print()
    print("Compare generated candidates against this baseline.")